# 🧹 AgentCore End-to-End Cleanup

This notebook provides a comprehensive cleanup process for all resources created during the AgentCore End-to-End tutorial.

## Overview

This cleanup process will remove:
- **Memory**: AgentCore Memory resources and stored data
- **Runtime**: Agent runtime instances and ECR repositories
- **Security**: Execution roles, and Authorization Provider resources
- **Observability**: CloudWatch log groups and streams
- **Local Files**: Generated configuration and code files

⚠️ **Important**: This cleanup is irreversible. Make sure you have saved any important data (if needed) before proceeding.

---

## Step 1: Import Required Dependencies

Load all necessary modules and helper functions for the cleanup process.

In [1]:
import json

from lab_helpers.lab2_memory import REGION
from lab_helpers.utils import (
    delete_agentcore_runtime_execution_role,
    delete_ssm_parameter,
    cleanup_cognito_resources,
    get_customer_support_secret,
    delete_customer_support_secret,
    agentcore_memory_cleanup,
    gateway_target_cleanup,
    runtime_resource_cleanup,
    delete_observability_resources,
    local_file_cleanup,
    get_ssm_parameter,
    policy_engine_cleanup
)

print("✅ Dependencies imported successfully")
print(f"🌍 Working in region: {REGION}")

✅ Dependencies imported successfully
🌍 Working in region: us-west-2


## Step 2: Clean Up Memory Resources

Remove AgentCore Memory resources and associated data.

In [2]:
print("🧠 Starting Memory cleanup...")
agentcore_memory_cleanup(get_ssm_parameter("/app/customersupport/agentcore/memory_id"))

🧠 Starting Memory cleanup...
✅ Successfully deleted memory: CustomerSupportMemory-OfyxiAA4d0


## Step 3: Clean Up Runtime Resources

Remove the AgentCore Runtime, ECR repository, and associated AWS resources.

In [3]:
print("🚀 Starting Runtime cleanup...")
runtime_resource_cleanup(
    get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
)

🚀 Starting Runtime cleanup...
  ✅ Agent runtime deleted: DELETING
  🗑️  Deleting ECR repository...
  ✅ ECR repository deleted: bedrock-agentcore-customer_support_agent


## Step 4: Clean Up Gateway Resources
Remove targets, Gateway

In [4]:
print("⚙️ Starting Policy Engine Cleanup...")
policy_engine_cleanup(get_ssm_parameter("/app/customersupport/agentcore/policy_engine_id"))

print("⚙️ Starting Gateway Cleanup...")
gateway_target_cleanup(get_ssm_parameter("/app/customersupport/agentcore/gateway_id"))

⚙️ Starting Policy Engine Cleanup...
🗑️  Deleting all policies for policy engine: customersupport_pe-i7pukqsewa
   Deleting policy: allow_policy-38s_2ozpwy
   ✅ Policy allow_policy-38s_2ozpwy deleted
   Deleting policy: deny_web_search-uone31dfcm
   ✅ Policy deny_web_search-uone31dfcm deleted
⏳ Waiting for policy deletions to propagate...
🗑️  Deleting policy engine: customersupport_pe-i7pukqsewa
✅ Policy engine customersupport_pe-i7pukqsewa deleted successfully
⚙️ Starting Gateway Cleanup...
🗑️  Deleting all targets for gateway: customersupport-gw-owicuwpou2
   Deleting target: JTKCB7CZUM
   ✅ Target JTKCB7CZUM deleted
⏳ Waiting for target deletions to propagate...
🗑️  Deleting gateway: customersupport-gw-owicuwpou2
✅ Gateway customersupport-gw-owicuwpou2 deleted successfully


## Step 5: Clean Up Security Resources

Remove execution roles, and authentication resources.

In [5]:
print("🛡️  Starting Security cleanup...")
try:
    # bedrock_client = boto3.client("bedrock", region_name=REGION)

    # Delete execution role
    print("  🗑️  Deleting AgentCore Runtime execution role...")
    delete_agentcore_runtime_execution_role()
    print("  ✅ Execution role deleted")

    # Delete SSM parameter
    print("  🗑️  Deleting SSM parameter...")
    delete_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    print("  ✅ SSM parameter deleted")

    # Clean up Cognito and secrets
    print("  🗑️  Cleaning up Cognito resources...")
    cs = json.loads(get_customer_support_secret())
    cleanup_cognito_resources(cs["pool_id"])
    print("  ✅ Cognito resources cleaned up")

    print("  🗑️  Deleting customer support secret...")
    delete_customer_support_secret()
    print("  ✅ Customer support secret deleted")

except Exception as e:
    print(f"  ⚠️  Error during security cleanup: {e}")

🛡️  Starting Security cleanup...
  🗑️  Deleting AgentCore Runtime execution role...
✅ Detached policy from role
✅ Deleted role: CustomerSupportAssistantBedrockAgentCoreRole-us-west-2
✅ Deleted policy: CustomerSupportAssistantBedrockAgentCorePolicy-us-west-2
  ✅ Execution role deleted
  🗑️  Deleting SSM parameter...
  ✅ SSM parameter deleted
  🗑️  Cleaning up Cognito resources...
Deleting app client: MCPServerPoolClient
Deleting user: testuser
Deleting user pool: us-west-2_FcV2DRH5T
Successfully cleaned up all Cognito resources
  ✅ Cognito resources cleaned up
  🗑️  Deleting customer support secret...
✅ Deleted secret!
  ✅ Customer support secret deleted


## Step 6: Clean Up Local Files

Remove generated configuration and code files from the local directory.

In [6]:
print("📁 Starting Local Files cleanup...")
local_file_cleanup()

📁 Starting Local Files cleanup...
  ✅ Deleted Dockerfile
  ✅ Deleted .dockerignore
  ✅ Deleted .bedrock_agentcore.yaml

📁 Successfully deleted 3 files
ℹ️  2 files were already missing: customer_support_agent.py, agent_runtime.py


## Step 7: Clean Up Observability Resources

Remove CloudWatch log groups and streams used for agent monitoring.

In [7]:
print("📊 Starting Observability cleanup...")

delete_observability_resources()

📊 Starting Observability cleanup...
  🗑️  Deleting log stream 'default'...
  ℹ️  Log stream 'default' doesn't exist
  🗑️  Deleting log group 'agents/customer-support-assistant-logs'...
  ℹ️  Log group 'agents/customer-support-assistant-logs' doesn't exist


## 🎉 Cleanup Complete!

All AgentCore resources have been cleaned up. Here's a summary of what was removed:

In [8]:
print("\n" + "=" * 60)
print("🧹 CLEANUP COMPLETED SUCCESSFULLY! 🧹")
print("=" * 60)
print()
print("📋 Resources cleaned up:")
print("  🧠 Memory: AgentCore Memory resources and data")
print("  🚀 Runtime: Agent runtime and ECR repository")
print("  🛡️ Security: Roles, and SSM secrets")
print("  📊 Observability: CloudWatch logs")
print("  📁 Files: Local configuration files")
print()
print("✨ Your AWS account is now clean and ready for new experiments!")
print("\nThank you for completing the AgentCore End-to-End tutorial! 🚀")


🧹 CLEANUP COMPLETED SUCCESSFULLY! 🧹

📋 Resources cleaned up:
  🧠 Memory: AgentCore Memory resources and data
  🚀 Runtime: Agent runtime and ECR repository
  🛡️ Security: Roles, and SSM secrets
  📊 Observability: CloudWatch logs
  📁 Files: Local configuration files

✨ Your AWS account is now clean and ready for new experiments!

Thank you for completing the AgentCore End-to-End tutorial! 🚀
